In [1]:
!pip install -q datasets sentence-transformers faiss-cpu rank_bm25 groq PyGithub

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/RAGBench_Results'
os.makedirs(SAVE_PATH, exist_ok=True)
print("Drive mounted. Results directory ready at", SAVE_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.1 MB/s eta 0:00:00
Mounted at /content/drive
Drive mounted. Results directory ready at /content/drive/MyDrive/RAGBench_Results


In [2]:
from google.colab import userdata

# ---------------- Groq API keys (from Colab Secrets, not hardcoded) ----------------
GROQ_KEYS = [
    userdata.get('GROQ_API_KEY_1'),
    userdata.get('GROQ_API_KEY_2'),
    userdata.get('GROQ_API_KEY_3'),
    userdata.get('GROQ_API_KEY_4'),
    userdata.get('GROQ_API_KEY_5'),
]
GROQ_KEYS = [k for k in GROQ_KEYS if k]
assert len(GROQ_KEYS) > 0, "Add your Groq keys to Colab Secrets (key icon, left sidebar) and enable notebook access."
current_key_index = 0

# ---------------- GitHub push (aiformetwosix-cpu, collaborator on veenulearns-lab repo) ----------------
REPO_OWNER = "veenulearns-lab"
REPO_NAME  = "RAGBench-Capstone-Batch26"
PUSH_TOKEN = userdata.get('GH_PAT_NEW')
assert PUSH_TOKEN, "Add aiformetwosix-cpu's PAT to Colab Secrets as GITHUB_TOKEN_AIFORMETWOSIX."

# ---------------- Judge model (fixed per your instruction) ----------------
JUDGE_MODEL = "llama-3.1-8b-instant"

# ---------------- Judge prompt: paste the FULL verbatim RAGBench Appendix 7.4 prompt ----------------
JUDGE_LLM_PROMPT = """
I asked someone to answer a question based on one or more
documents. Your task is to review their response and assess whether or not each
sentence in that response is supported by text in the documents. And if so, which
sentences in the documents provide that support. You will also tell me which
of the documents contain useful information for answering the question, and
which of the documents the answer was sourced from.

Here are the documents, each of which is split into sentences. Alongside each
sentence is associated key, such as '0a.' or '0b.' that you can use to refer
to it:

```
{documents}
```

The question was:
```
{question}
```

Here is their response, split into sentences. Alongside each sentence is
associated key, such as 'a.' or 'b.' that you can use to refer to it. Note
that these keys are unique to the response, and are not related to the keys
in the documents:

```
{answer}
```

You must respond with a JSON object matching this schema:

{{
  "relevance_explanation": string,
  "all_relevant_sentence_keys": [string],
  "overall_supported_explanation": string,
  "overall_supported": boolean,
  "sentence_support_information": [
    {{
      "response_sentence_key": string,
      "explanation": string,
      "supporting_sentence_keys": [string],
      "fully_supported": boolean
    }}
  ],
  "all_utilized_sentence_keys": [string]
}}

The relevance_explanation field is a string explaining which documents
contain useful information for answering the question. Walk through the
information in the documents step by step and how it is useful for
answering the question.

The all_relevant_sentence_keys field is a list of all document sentence
keys (e.g. '0a') that are relevant to the question. Include every sentence
that is useful and relevant to the question, even if it was not used in the
response, or if only parts of the sentence are useful. Base this judgement
only on the documents and the question -- ignore the response entirely when
deciding relevance. Leave out sentences that could be removed from the
document without affecting someone's ability to answer the question.

The overall_supported_explanation field is a string explaining why the
response *as a whole* is or is not supported by the documents. Walk through
each claim in the response individually and assess its support (or lack of
support) in the documents one at a time, before drawing any conclusion about
the response as a whole.

The overall_supported field is a boolean reflecting the conclusion you
reached at the end of overall_supported_explanation: whether the response as
a whole is supported by the documents.

The sentence_support_information field is a list of objects, one for each sentence
in the response. Each object MUST have the following fields:
- response_sentence_key: a string identifying the sentence in the response. This
key is the same as the one used in the response above.- explanation: a string
explaining why the sentence is or is not supported by the documents.
- supporting_sentence_keys: keys (e.g. ’0a’) of sentences from the documents that
support the response sentence. If the sentence is not supported, this list MUST
be empty. If the sentence is supported, this list MUST contain one or more keys.
In special cases where the sentence is supported, but not by any specific sentence,
you can use the string "supported_without_sentence" to indicate that the sentence
is generally supported by the documents. Consider cases where the sentence is
expressing inability to answer the question due to lack of relevant information
in the provided contex as "supported_without_sentence". In cases
where the sentence is making a general statement (e.g. outlining the steps to produce
an answer, or summarizing previously stated sentences, or a transition sentence), use
the sting "general".In cases where the sentence is correctly stating a well-known fact,
like a mathematical formula, use the string "well_known_fact". In cases where the
sentence is performing numerical reasoning (e.g. addition, multiplication), use
the string "numerical_reasoning".
- fully_supported: a boolean indicating whether the sentence is fully supported by
the documents.
  - This value should reflect the conclusion  you drew at the end of your step-by-step
    breakdown in explanation.
  - If supporting_sentence_keys is an empty list, then fully_supported must be false.
  - Otherwise, use fully_supported to clarify whether everything in the response
  sentence is fully supported by the document text indicated in supporting_sentence_keys
  (fully_supported = true), or whether the sentence is only partially or incompletely
  supported by that document text (fully_supported = false).

The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’0a’) that
were used to construct the answer. Include every sentence that either directly supported
the answer, or was implicitly used to construct the answer, even if it was not used
in its entirety. Omit sentences that were not used, and could have been removed from
the documents without affecting the answer.

You must respond with a valid JSON string. Use escapes for quotes, e.g. ‘\\"‘, and
newlines, e.g. ‘\\n‘. Do not write anything before or after the JSON string. Do not
wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.

As a reminder: your task is to review the response and assess which documents contain
useful information pertaining to the question, and how each sentence in the response
is supported by the text in the documents.
"""
assert "PASTE" not in JUDGE_LLM_PROMPT, "You forgot to paste the verbatim judge prompt!"

print(f"Judge model: {JUDGE_MODEL}")
print(f"{len(GROQ_KEYS)} Groq keys loaded.")
print(f"GitHub push target: {REPO_OWNER}/{REPO_NAME} (as aiformetwosix-cpu)")


Judge model: llama-3.1-8b-instant
5 Groq keys loaded.
GitHub push target: veenulearns-lab/RAGBench-Capstone-Batch26 (as aiformetwosix-cpu)


In [4]:
#3 - core functions
import re, json, time, itertools
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, models
import faiss
from rank_bm25 import BM25Okapi
from groq import Groq
from sklearn.metrics import roc_auc_score, mean_squared_error

# ================= Groq key rotation =================
def get_rotated_client():
    return Groq(api_key=GROQ_KEYS[current_key_index], timeout=30.0)

def rotate_key():
    global current_key_index
    current_key_index = (current_key_index + 1) % len(GROQ_KEYS)
    print(f"    🔄 Switched to key {current_key_index + 1}")
    return get_rotated_client()

# ================= Embedder cache (incl. FinBERT via mean pooling) =================
_embedder_cache = {}

def build_finbert_embedder():
    word_embedding_model = models.Transformer("ProsusAI/finbert")
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
    return SentenceTransformer(modules=[word_embedding_model, pooling_model])

def get_embedder(embed_model_name):
    if embed_model_name not in _embedder_cache:
        print(f"  Loading embedder: {embed_model_name} ...")
        if embed_model_name == "ProsusAI/finbert":
            _embedder_cache[embed_model_name] = build_finbert_embedder()
        else:
            _embedder_cache[embed_model_name] = SentenceTransformer(embed_model_name)
    return _embedder_cache[embed_model_name]

embedder = get_embedder("all-MiniLM-L6-v2")  # default, matches Phase 1 baseline

# ================= Document formatting (for judge) =================
def format_documents_with_keys(documents_sentences):
    formatted = ""
    for doc_sentences in documents_sentences:
        for key, sentence in doc_sentences:
            formatted += f"{key}. {sentence}\n"
        formatted += "\n"
    return formatted.strip()

def format_response_with_keys(response):
    sentences = re.split(r'(?<=[.!?])\s+', response.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    formatted = ""
    for idx, sentence in enumerate(sentences):
        key = chr(97 + idx)
        formatted += f"{key}. {sentence}\n"
    return formatted.strip(), [chr(97 + i) for i in range(len(sentences))]

# ================= Chunking =================
def build_large_semantic_chunks(documents_sentences, max_chars=1100):
    """Merge consecutive sentences into ~1100-char chunks for retrieval only.
    Labeled 'metadata-aware' per the domain matrix: keeps numeric figures with
    their labels together, matching the CP-2 finance finding."""
    chunk_texts = []
    for doc_sentences in documents_sentences:
        buf = ""
        for key, sentence in doc_sentences:
            buf += " " + sentence
            if len(buf) >= max_chars:
                chunk_texts.append(buf.strip())
                buf = ""
        if buf:
            chunk_texts.append(buf.strip())
    return chunk_texts

# ================= Generator (flexible: prompt style + model) =================
def generate_response_flexible(question, retrieved_docs, prompt_style="full", generator_model="llama-3.1-8b-instant"):
    global current_key_index
    context = "\n\n".join([f"Document {i+1}:\n{doc}" for i, doc in enumerate(retrieved_docs)])
    if prompt_style == "minimal":
        prompt = f"Use the following context to answer: {context} Question: {question}"
    else:
        prompt = f"Answer ONLY from context. If insufficient, say cannot answer. {context}\nQuestion: {question}"
    for attempt in range(len(GROQ_KEYS) + 1):
        try:
            response = get_rotated_client().chat.completions.create(
                model=generator_model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=256,
                temperature=0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if '429' in str(e):
                rotate_key()
                time.sleep(5)
            else:
                raise e
    return None

# ================= Judge (fixed model, verbatim prompt) =================
def run_judge_rotated(question, documents_sentences, response):
    global current_key_index
    formatted_docs = format_documents_with_keys(documents_sentences)
    formatted_response, _ = format_response_with_keys(response)
    all_sentence_keys = []
    for doc_sentences in documents_sentences:
        for key, sentence in doc_sentences:
            all_sentence_keys.append(key)

    if "{documents}" in JUDGE_LLM_PROMPT or "{question}" in JUDGE_LLM_PROMPT:
        prompt = JUDGE_LLM_PROMPT.format(
            documents=formatted_docs, question=question, answer=formatted_response
        )
    else:
        prompt = (
            f"{JUDGE_LLM_PROMPT}\n\n"
            f"Documents:\n'''\n{formatted_docs}\n'''\n\n"
            f"Question:\n'''\n{question}\n'''\n\n"
            f"Response:\n'''\n{formatted_response}\n'''"
        )

    for attempt in range(len(GROQ_KEYS) + 1):
        try:
            response_judge = get_rotated_client().chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2048,
                temperature=0
            )
            raw = response_judge.choices[0].message.content.strip()
            if "```json" in raw:
                raw = raw.split("```json")[1].split("```")[0].strip()
            elif "```" in raw:
                raw = raw.split("```")[1].split("```")[0].strip()
            result = json.loads(raw)
            return result, all_sentence_keys
        except Exception as e:
            if '429' in str(e):
                rotate_key()
                time.sleep(5)
            else:
                return None, all_sentence_keys
    return None, all_sentence_keys

# ================= TRACe metrics (independent implementation, no external libs) =================
# FIXED: clip judge-returned keys to the real document key set before computing ratios,
# so hallucinated/duplicate keys from the judge can't push a ratio above 1.0.
def compute_trace_metrics(judge_output, all_sentence_keys):
    if not judge_output:
        return {'context_relevance': 0.0, 'context_utilization': 0.0, 'completeness': 0.0, 'adherence': False}

    all_keys_set = set(all_sentence_keys)
    all_relevant = set(judge_output.get('all_relevant_sentence_keys', [])) & all_keys_set
    all_utilized = set(judge_output.get('all_utilized_sentence_keys', [])) & all_keys_set

    total = len(all_keys_set)
    context_relevance = len(all_relevant) / total if total > 0 else 0.0
    context_utilization = len(all_utilized) / total if total > 0 else 0.0
    completeness = (len(all_relevant & all_utilized) / len(all_relevant)) if all_relevant else 0.0

    sentence_support = judge_output.get('sentence_support_information', [])
    if sentence_support:
        adherence = all(s.get('fully_supported', False) for s in sentence_support)
    else:
        adherence = judge_output.get('overall_supported', False)

    return {
        'context_relevance': round(context_relevance, 4),
        'context_utilization': round(context_utilization, 4),
        'completeness': round(completeness, 4),
        'adherence': adherence,
    }

# ================= Retrieval (dense / hybrid RRF) =================
def retrieve_top_k_hybrid(question, documents, index, active_embedder, k=3):
    q_emb = np.array(active_embedder.encode([question])).astype('float32')
    _, dense_idx = index.search(q_emb, max(k * 2, 10))
    dense_idx = [i for i in dense_idx[0] if i != -1]

    tokenized = [d.lower().split() for d in documents]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = bm25.get_scores(question.lower().split())
    bm25_idx = list(np.argsort(bm25_scores)[::-1][:max(k * 2, 10)])

    rrf = {}
    for rank, i in enumerate(dense_idx):
        rrf[i] = rrf.get(i, 0) + 1.0 / (rank + 60)
    for rank, i in enumerate(bm25_idx):
        rrf[i] = rrf.get(i, 0) + 1.0 / (rank + 60)
    ranked = sorted(rrf.items(), key=lambda x: -x[1])[:k]
    return [documents[i] for i, _ in ranked]

# ================= Flexible full pipeline =================
def run_pipeline_flexible(example, k=3, prompt_style="full", chunking="whole_doc",
                           retrieval="dense", embed_model_name="all-MiniLM-L6-v2",
                           generator_model="llama-3.1-8b-instant"):
    question = example['question']
    documents = example['documents']
    documents_sentences = example['documents_sentences']

    corpus = build_large_semantic_chunks(documents_sentences, max_chars=1100) if chunking == "large_semantic" else documents

    active_embedder = get_embedder(embed_model_name)
    doc_embeddings = np.array(active_embedder.encode(corpus, show_progress_bar=False)).astype('float32')
    index = faiss.IndexFlatL2(doc_embeddings.shape[1])
    index.add(doc_embeddings)

    if retrieval == "hybrid":
        top_k_docs = retrieve_top_k_hybrid(question, corpus, index, active_embedder, k=k)
    else:
        q_emb = np.array(active_embedder.encode([question])).astype('float32')
        _, idxs = index.search(q_emb, k)
        top_k_docs = [corpus[i] for i in idxs[0] if i != -1]

    our_response = generate_response_flexible(question, top_k_docs, prompt_style=prompt_style, generator_model=generator_model)
    if our_response is None:
        return None

    # Judge always evaluates against the FULL original documents_sentences,
    # regardless of the retrieval/chunking strategy used to produce the answer.
    judge_output, all_keys = run_judge_rotated(question, documents_sentences, our_response)
    if judge_output is None:
        return None

    return {'metrics': compute_trace_metrics(judge_output, all_keys)}

print("✅ All functions loaded (secrets-based keys, FinBERT-capable, multi-LLM generator, fixed TRACe metrics).")

/tmp/ipykernel_31000/4036911115.py:6: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models


  Loading embedder: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All functions loaded (secrets-based keys, FinBERT-capable, multi-LLM generator, fixed TRACe metrics).


In [5]:
#BASELINE

BASELINE_25 = None  # will be computed fresh below

baseline_scores = run_experiment("verbatim", finqa_dataset, eval_indices_25, SAVE_PATH,
                                   k=3, prompt_style="full", chunking="whole_doc",
                                   retrieval="dense", embed_model_name="all-MiniLM-L6-v2",
                                   generator_model="llama-3.1-8b-instant")
BASELINE_25 = baseline_scores
print("\nFresh baseline (corrected metrics):")
for k, v in BASELINE_25.items():
    print(f"  {k}: {v}")

Resuming verbatim from 20 saved examples
    🔄 Switched to key 2
verbatim: Rel 0.2915 | Util 0.1183 | Comp 0.6335 | Adh 0.5379

Fresh baseline (corrected metrics):
  rel_rmse: 0.2915342913901166
  util_rmse: 0.11829487805739543
  comp_rmse: 0.6335167886927352
  adh_aucroc: 0.537878787878788


In [5]:
# 5================= Dataset + generic experiment runner =================
finqa_dataset = load_dataset("rungalileo/ragbench", "finqa", split="test")
eval_indices_25 = list(range(25))  # SAME 25 indices as the baseline, for a fair comparison

def run_experiment(exp_name, dataset, eval_indices, save_path, **pipeline_kwargs):
    progress_file = f'{save_path}/cp1_finance_{exp_name}_progress.csv'
    final_file    = f'{save_path}/cp1_finance_{exp_name}_25samples.csv'

    if os.path.exists(progress_file):
        df_existing = pd.read_csv(progress_file)
        results = df_existing.to_dict('records')
        completed = set(df_existing['idx'].tolist())
        print(f"Resuming {exp_name} from {len(results)} saved examples")
    else:
        results, completed = [], set()

    for i in eval_indices:
        if i in completed:
            continue
        print(f"  → attempting idx {i}...")   # ADD THIS
        try:
            example = dataset[i]
            result = run_pipeline_flexible(example, **pipeline_kwargs)
            if result is not None:
                results.append({
                    'idx': i,
                    'our_relevance': result['metrics']['context_relevance'],
                    'our_utilization': result['metrics']['context_utilization'],
                    'our_completeness': result['metrics']['completeness'],
                    'our_adherence': 1 if result['metrics']['adherence'] else 0,
                    'ref_relevance': example['relevance_score'],
                    'ref_utilization': example['utilization_score'],
                    'ref_completeness': example['completeness_score'],
                    'ref_adherence': 1 if example['adherence_score'] else 0,
                })
                completed.add(i)
        except Exception as e:
            if '429' in str(e):
                print(f"  Rate limit at {i}. Waiting 30s...")
                time.sleep(30)
            else:
                print(f"  \u26A0\uFE0F {i}: {str(e)[:80]}")
        if len(results) % 5 == 0 and len(results) > 0:
            pd.DataFrame(results).to_csv(progress_file, index=False)
        time.sleep(1)

    df = pd.DataFrame(results)
    df.to_csv(final_file, index=False)
    if len(df) > 5:
        rel = np.sqrt(mean_squared_error(df['ref_relevance'], df['our_relevance']))
        util = np.sqrt(mean_squared_error(df['ref_utilization'], df['our_utilization']))
        comp = np.sqrt(mean_squared_error(df['ref_completeness'], df['our_completeness']))
        try:
            adh = roc_auc_score(df['ref_adherence'], df['our_adherence'])
        except Exception:
            adh = 0.5
        scores = {'rel_rmse': rel, 'util_rmse': util, 'comp_rmse': comp, 'adh_aucroc': adh}
        print(f"{exp_name}: Rel {rel:.4f} | Util {util:.4f} | Comp {comp:.4f} | Adh {adh:.4f}")
        return scores
    print(f"{exp_name}: not enough completed examples to score")
    return None

print("Dataset loaded:", len(finqa_dataset), "examples. Experiment runner ready.")


README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

finqa/train-00000-of-00001.parquet:   0%|          | 0.00/61.1M [00:00<?, ?B/s]

finqa/validation-00000-of-00001.parquet:   0%|          | 0.00/5.94M [00:00<?, ?B/s]

finqa/test-00000-of-00001.parquet:   0%|          | 0.00/8.94M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12502 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1766 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2294 [00:00<?, ? examples/s]

Dataset loaded: 2294 examples. Experiment runner ready.


In [11]:
experiments_25 = {}
print("Exp 1: k=3 vs k=5")
experiments_25["exp1_k5"] = run_experiment("exp1_k5", finqa_dataset, eval_indices_25, SAVE_PATH, k=5)

Exp 1: k=3 vs k=5
Resuming exp1_k5 from 20 saved examples
  → attempting idx 20...
  → attempting idx 21...
  → attempting idx 22...
  → attempting idx 23...
  → attempting idx 24...
exp1_k5: Rel 0.2903 | Util 0.1143 | Comp 0.6473 | Adh 0.5379


In [12]:
# ================= Phase 2: Experiments (25 samples each, same eval_indices_25) =================
#experiments_25 = {}

#print("Exp 1: k=3 vs k=5")
#experiments_25["exp1_k5"] = run_experiment("exp1_k5", finqa_dataset, eval_indices_25, SAVE_PATH, k=5)

print("\nExp 2: Generator prompt minimal vs full")
experiments_25["exp2_minimal"] = run_experiment("exp2_minimal", finqa_dataset, eval_indices_25, SAVE_PATH, prompt_style="minimal")

print("\nExp 3: Chunking whole-doc vs large semantic / metadata-aware (~1100c)")
experiments_25["exp3_chunk"] = run_experiment("exp3_chunk", finqa_dataset, eval_indices_25, SAVE_PATH, chunking="large_semantic")

print("\nExp 4: Embedding MiniLM vs BGE-large")
experiments_25["exp4_bge"] = run_experiment("exp4_bge", finqa_dataset, eval_indices_25, SAVE_PATH, embed_model_name="BAAI/bge-large-en-v1.5")

print("\nExp 5: Retrieval dense vs hybrid BM25+dense (RRF)")
experiments_25["exp5_hybrid"] = run_experiment("exp5_hybrid", finqa_dataset, eval_indices_25, SAVE_PATH, retrieval="hybrid")

print("\nExp 6: Embedding MiniLM vs FinBERT (domain-specific, mandatory per capstone doc)")
experiments_25["exp6_finbert"] = run_experiment("exp6_finbert", finqa_dataset, eval_indices_25, SAVE_PATH, embed_model_name="ProsusAI/finbert")

print("\nExp 7: Generator LLM 8B vs 70B (moving toward the 4-5 LLM requirement)")
experiments_25["exp7_llama70b_gen"] = run_experiment("exp7_llama70b_gen", finqa_dataset, eval_indices_25, SAVE_PATH, generator_model="llama-3.3-70b-versatile")

print("\nExp 8: Generator LLM 8B vs qwen3-32b")
experiments_25["exp8_qwen32b_gen"] = run_experiment("exp8_qwen32b_gen", finqa_dataset, eval_indices_25, SAVE_PATH, generator_model="qwen/qwen3-32b")

print("\nAll Phase 2 scores:")
for name, s in experiments_25.items():
    print(f"  {name}: {s}")



Exp 2: Generator prompt minimal vs full
  → attempting idx 0...
  → attempting idx 1...
  → attempting idx 2...
  → attempting idx 3...
  → attempting idx 4...
  → attempting idx 5...
  → attempting idx 6...
  → attempting idx 7...
    🔄 Switched to key 3
  → attempting idx 8...
  → attempting idx 9...
  → attempting idx 10...
  → attempting idx 11...
  → attempting idx 12...
    🔄 Switched to key 4
  → attempting idx 13...
  → attempting idx 14...
  → attempting idx 15...
  → attempting idx 16...
  → attempting idx 17...
  → attempting idx 18...
  → attempting idx 19...
  → attempting idx 20...
  → attempting idx 21...
  → attempting idx 22...
  → attempting idx 23...
  → attempting idx 24...
exp2_minimal: Rel 0.3947 | Util 0.1732 | Comp 0.6702 | Adh 0.6818

Exp 3: Chunking whole-doc vs large semantic / metadata-aware (~1100c)
  → attempting idx 0...
  → attempting idx 1...
  → attempting idx 2...
  → attempting idx 3...
  → attempting idx 4...
    🔄 Switched to key 5
  → attempting 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/tmp/ipykernel_12191/4036911115.py:27: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")


  → attempting idx 1...
  → attempting idx 2...
  → attempting idx 3...
  → attempting idx 4...
    🔄 Switched to key 5
  → attempting idx 5...
  → attempting idx 6...
  → attempting idx 7...
  → attempting idx 8...
  → attempting idx 9...
  → attempting idx 10...
  → attempting idx 11...
  → attempting idx 12...
  → attempting idx 13...
  → attempting idx 14...
  → attempting idx 15...
  → attempting idx 16...
  → attempting idx 17...
  → attempting idx 18...
  → attempting idx 19...
  → attempting idx 20...
    🔄 Switched to key 1
  → attempting idx 21...
  → attempting idx 22...
  → attempting idx 23...
  → attempting idx 24...
exp6_finbert: Rel 0.3198 | Util 0.0856 | Comp 0.6774 | Adh 0.5152

Exp 7: Generator LLM 8B vs 70B (moving toward the 4-5 LLM requirement)
  → attempting idx 0...
  → attempting idx 1...
  → attempting idx 2...
  → attempting idx 3...
  → attempting idx 4...
  → attempting idx 5...
  → attempting idx 6...
  → attempting idx 7...
  → attempting idx 8...
  → att

In [6]:
# change in generator model
BEST_SETTINGS = dict(
    k=3,
    prompt_style="full",
    chunking="whole_doc",
    embed_model_name="all-MiniLM-L6-v2",
    retrieval="dense",
    generator_model="llama-3.3-70b-versatile",
)

print("Running Phase 3: Best combination (25 samples)...")
best_scores = run_experiment("best_combo", finqa_dataset, eval_indices_25, SAVE_PATH, **BEST_SETTINGS)
print("\nPhase 3 — Best Combination:")
for k, v in best_scores.items():
    print(f"  {k}: {v}")

Running Phase 3: Best combination (25 samples)...
  → attempting idx 0...
  → attempting idx 1...
  → attempting idx 2...
  → attempting idx 3...
  → attempting idx 4...
  → attempting idx 5...
  → attempting idx 6...
  → attempting idx 7...
  → attempting idx 8...
  → attempting idx 9...
  → attempting idx 10...
  → attempting idx 11...
  → attempting idx 12...
  → attempting idx 13...
  → attempting idx 14...
  → attempting idx 15...
  → attempting idx 16...
  → attempting idx 17...
  → attempting idx 18...
    🔄 Switched to key 2
  → attempting idx 19...
  → attempting idx 20...
  → attempting idx 21...
  → attempting idx 22...
  → attempting idx 23...
  → attempting idx 24...
best_combo: Rel 0.4379 | Util 0.1403 | Comp 0.7020 | Adh 0.7500

Phase 3 — Best Combination:
  rel_rmse: 0.4379358292359489
  util_rmse: 0.14034221569799982
  comp_rmse: 0.7020096963031948
  adh_aucroc: 0.75


In [ ]:
# ================= Summary table vs baseline =================
print(f"{'Experiment':<20}{'Rel RMSE':<12}{'Util RMSE':<12}{'Comp RMSE':<12}{'Adh AUCROC'}")
print(f"{'Baseline':<20}{BASELINE_25['rel_rmse']:<12.4f}{BASELINE_25['util_rmse']:<12.4f}{BASELINE_25['comp_rmse']:<12.4f}{BASELINE_25['adh_aucroc']:.4f}")
for name, s in experiments_25.items():
    if s:
        print(f"{name:<20}{s['rel_rmse']:<12.4f}{s['util_rmse']:<12.4f}{s['comp_rmse']:<12.4f}{s['adh_aucroc']:.4f}")


In [ ]:
# ================= Push Phase 2 results to GitHub (aiformetwosix-cpu) =================
import shutil, subprocess

repo_dir = "/content/repo"
if not os.path.exists(f"{repo_dir}/.git"):
    subprocess.run(["rm", "-rf", repo_dir])
    subprocess.run(["git", "clone", f"https://{PUSH_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git", repo_dir], check=True)

os.makedirs(f"{repo_dir}/results", exist_ok=True)

exp_files = [f"cp1_finance_{name}_25samples.csv" for name in experiments_25.keys()]
for fname in exp_files:
    local_path = f"{SAVE_PATH}/{fname}"
    if os.path.exists(local_path):
        shutil.copy(local_path, f"{repo_dir}/results/{fname}")
    else:
        print(f"  Skipped (not found): {local_path}")

subprocess.run(["git", "-C", repo_dir, "config", "user.email", "aiformetwosix-cpu@users.noreply.github.com"], check=True)
subprocess.run(["git", "-C", repo_dir, "config", "user.name", "aiformetwosix-cpu"], check=True)
subprocess.run(["git", "-C", repo_dir, "add", "."], check=True)
subprocess.run(["git", "-C", repo_dir, "status"])
subprocess.run(["git", "-C", repo_dir, "commit", "-m",
                 "[Veenu] CP-3: Finance Phase 2 experiments (25 samples) - FinBERT + multi-LLM generator added per capstone compliance review"],
                check=False)
subprocess.run(["git", "-C", repo_dir, "push", "origin", "main"], check=True)

print("\u2705 Phase 2 experiment results pushed to GitHub!")
